In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# Load and preprocess MNIST data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Expand dims to match CNN input shape (batch, height, width, channels)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# Split training into train/val
x_val = x_train[-5000:]
y_val = y_train[-5000:]
x_train = x_train[:-5000]
y_train = y_train[:-5000]

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    shear_range=0.1
)
datagen.fit(x_train)

# Build the improved model
model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.GaussianNoise(0.1),

    layers.Conv2D(32, 3, activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.3),

    layers.Conv2D(64, 3, activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),
    layers.Dropout(0.3),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-3)),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
checkpoint_cb = callbacks.ModelCheckpoint("best_model.keras", save_best_only=True)
early_stop_cb = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
reduce_lr_cb = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)

# Train the model
history = model.fit(
    datagen.flow(x_train, y_train, batch_size=32),
    epochs=30,
    validation_data=(x_val, y_val),
    callbacks=[checkpoint_cb, early_stop_cb, reduce_lr_cb]
)

# Evaluate on clean test data
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"✅ Final Test Accuracy: {test_acc:.4f}")

# Predict and print classification report
y_pred = np.argmax(model.predict(x_test), axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Save full model
model.save("mnist_final_model.keras")
print("✅ Full model saved to 'mnist_final_model.keras'")

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open("mnist_model.tflite", "wb") as f:
    f.write(tflite_model)
print("✅ Model converted and saved as 'mnist_model.tflite'")

# Optionally test on noisy version (production-like)
x_test_noisy = x_test + np.random.normal(0, 0.05, x_test.shape)
x_test_noisy = np.clip(x_test_noisy, 0.0, 1.0)
noisy_loss, noisy_acc = model.evaluate(x_test_noisy, y_test)
print(f"📦 Accuracy on Noisy (Realistic) Test Set: {noisy_acc:.4f}")

2025-06-02 11:18:32.024874: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748863112.234780      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748863112.290484      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


I0000 00:00:1748863126.806492      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1748863126.807174      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/30


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
I0000 00:00:1748863134.453112      60 service.cc:148] XLA service 0x793d50007730 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1748863134.453995      60 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1748863134.454013      60 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1748863135.006790      60 cuda_dnn.cc:529] Loaded cuDNN version 90300


  17/1719 ━━━━━━━━━━━━━━━━━━━━ 16s 10ms/step - accuracy: 0.1292 - loss: 3.3281

I0000 00:00:1748863140.426011      60 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1719/1719 ━━━━━━━━━━━━━━━━━━━━ 36s 15ms/step - accuracy: 0.6809 - loss: 1.2210 - val_accuracy: 0.9838 - val_loss: 0.2543 - learning_rate: 0.0010
Epoch 2/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - accuracy: 0.9439 - loss: 0.3685 - val_accuracy: 0.9850 - val_loss: 0.2069 - learning_rate: 0.0010
Epoch 3/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - accuracy: 0.9595 - loss: 0.2881 - val_accuracy: 0.9850 - val_loss: 0.1829 - learning_rate: 0.0010
Epoch 4/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - accuracy: 0.9613 - loss: 0.2566 - val_accuracy: 0.9876 - val_loss: 0.1705 - learning_rate: 0.0010
Epoch 5/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - accuracy: 0.9625 - loss: 0.2456 - val_accuracy: 0.9888 - val_loss: 0.1651 - learning_rate: 0.0010
Epoch 6/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - accuracy: 0.9649 - loss: 0.2349 - val_accuracy: 0.9902 - val_loss: 0.1485 - learning_rate: 0.0010
Epoch 7/30
1719/1719 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - accuracy: 0.9654 

W0000 00:00:1748863682.857259      19 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1748863682.857284      19 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1748863682.874534      19 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled


✅ Model converted and saved as 'mnist_model.tflite'
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9944 - loss: 0.0500
📦 Accuracy on Noisy (Realistic) Test Set: 0.9956
